<a href="https://colab.research.google.com/github/asantibane-star/challenge.trading.itau/blob/main/Limpiador_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
import pandas as pd
import holidays

ARCHIVO_BRUTO = 'https://raw.githubusercontent.com/asantibane-star/challenge.trading.itau/main/base_usdclp_actualizada.xlsx'
ARCHIVO_MODELO = 'base_modelo.xlsx'
INICIO = '2004-01-01'

raw = pd.read_excel(ARCHIVO_BRUTO, sheet_name='Días hábiles').set_index('fecha').sort_index()
raw = raw[raw.index >= INICIO]
print('Base bruta desde 2004:', raw.shape)

Base bruta desde 2004: (5962, 77)


In [2]:
# =============================================================================
# 1. Filtrado temporal: solo días en que operó el mercado cambiario chileno
# =============================================================================
feriados_cl = holidays.Chile(years=range(2004, raw.index.year.max() + 1))
es_feriado = np.array([f in feriados_cl for f in raw.index])
es_31dic = (raw.index.month == 12) & (raw.index.day == 31)      # feriado bancario, no figura en el calendario oficial
sin_usdclp = raw['USDCLP Curncy|PX_LAST'].isna().values             # incluye el hueco de 2011-2012
eliminar = es_feriado | es_31dic | sin_usdclp
print('Feriados eliminados:', int((es_feriado | es_31dic).sum()), '| días sin USDCLP eliminados:', int((sin_usdclp & ~(es_feriado | es_31dic)).sum()))

# Se excluyen las variables mensuales (IPC, IMACEC, desempleo) y las columnas vacías desde 2004
mensuales = [c for c in raw.columns if any(t in c for t in ['CLINNSYO', 'CLINNSMO', 'CLIMMOMS', 'CLIMYOYN', 'CHUETOTL'])]
b = raw[~eliminar].drop(columns=mensuales)
b = b.drop(columns=[c for c in b.columns if b[c].isna().all()])

# Días corridos desde la observación anterior: identifica retornos que cruzan huecos largos
b['dias_desde_obs_anterior'] = b.index.to_series().diff().dt.days
print('Base limpia:', b.shape)

Feriados eliminados: 265 | días sin USDCLP eliminados: 157
Base limpia: (5540, 67)


In [3]:
# =============================================================================
# 2. Empalmes, rezagos y relleno de niveles
# =============================================================================
series = [c for c in b.columns if '|' in c]
b[series] = b[series].ffill(limit=3)            # mercado externo cerrado: se mantiene el último nivel, máximo 3 días
salto_ok = b['dias_desde_obs_anterior'] <= 5    # retornos que cruzan huecos de más de 5 días no se usan

O, H, L, Cl = [b['USDCLP Curncy|' + k] for k in ['PX_OPEN', 'PX_HIGH', 'PX_LOW', 'PX_LAST']]
H_aj = pd.concat([H, O, Cl], axis=1).max(axis=1)   # en 17 jornadas el rango no contenía apertura o cierre
L_aj = pd.concat([L, O, Cl], axis=1).min(axis=1)

tasa_usd = b['SOFRRATE Index|PX_LAST'].fillna(b['FEDL01 Index|PX_LAST']).shift(1)   # se publica al día siguiente
flujo_off = b['CLPBODMB Index|PX_LAST'].fillna(b['CLPBODM Index|PX_LAST'])           # empalme series BCCh
flujo_loc = b['CLPBLDMB Index|PX_LAST'].fillna(b['CLPBLDM Index|PX_LAST'])

def ret(col):                       # retorno logarítmico
    return np.log(b[col]).diff().where(salto_ok)

def dif(col):                       # primera diferencia
    return b[col].diff().where(salto_ok)

def momento(n):                     # retorno acumulado de n días del USD/CLP
    return np.log(Cl).diff(n).where(salto_ok.rolling(n).min() == 1)

oo = np.log(O).diff().where(salto_ok)   # retorno de apertura a apertura

In [4]:
# =============================================================================
# 3. Variables del modelo (todas conocidas al cierre del día t)
# =============================================================================
X = pd.DataFrame({
    '1 USDCLP retorno cierre': ret('USDCLP Curncy|PX_LAST'),
    '3 USDCLP momentum 5d': momento(5),
    '5 USDCLP salto apertura': np.log(O / Cl.shift(1)).where(salto_ok),
    '6 USDCLP rango log(H/L)': np.log(H_aj / L_aj),
    '7 USDCLP momentum 20d': momento(20),
    '8 DXY': ret('DXY Curncy|PX_LAST'),
    '11 USDCNY': ret('USDCNY Curncy|PX_LAST'),
    '13 USDJPY': ret('USDJPY Curncy|PX_LAST'),
    '14 USDMXN': ret('USDMXN Curncy|PX_LAST'),
    '17 Vol FX EM (JPM)': dif('JPMVXYEM Index|PX_LAST'),
    '18 USDPEN': ret('USDPEN Curncy|PX_LAST'),
    '19 USDCOP': ret('USDCOP Curncy|PX_LAST'),
    '21 Cobre LME': ret('LMCADS03 Comdty|PX_LAST'),
    '24 BCOM': ret('BCOM Index|PX_LAST'),
    '23 Brent': ret('CO1 Comdty|PX_LAST'),
    '26 Oro': ret('XAU Curncy|PX_LAST'),
    '28 MSCI EM': ret('MXEF Index|PX_LAST'),
    '29 SPX': ret('SPX Index|PX_LAST'),
    '31 Shanghai': ret('SHCOMP Index|PX_LAST'),
    '33 MOVE (dif)': dif('MOVE Index|PX_LAST'),
    '34 VIX (nivel)': b['VIX Index|PX_LAST'],
    '35 UST 2Y': dif('USGG2YR Index|PX_LAST'),
    '37 UST 3M': dif('USGG3M Index|PX_LAST'),
    '40 Pendiente UST 10Y-3M (nivel)': b['USGG10YR Index|PX_LAST'] - b['USGG3M Index|PX_LAST'],
    '41 Swap CLP 1Y': dif('CHSWP1 Curncy|PX_LAST'),
    '45 TPM': dif('CHOVCHOV Index|PX_LAST'),
    '48 Swap UF 2Y': dif('CHSWC2 Curncy|PX_LAST'),
    '52 Carry CLP-USD (nivel)': b['CHOVCHOV Index|PX_LAST'] - tasa_usd,
    '53 Pendiente swap CLP 5Y-2Y (nivel)': b['CHSWP5 Curncy|PX_LAST'] - b['CHSWP2 Curncy|PX_LAST'],
    '54 NDF pts 3M': dif('CHN3M Curncy|PX_LAST'),
    '57 NDF tasa impl 3M': dif('CHNI3M Curncy|PX_LAST'),
    '59 Vol impl CLP 1M (dif)': dif('USDCLPV1M Curncy|PX_LAST'),
    '62 NDF pts 1M': dif('CHN1M Curncy|PX_LAST'),
    '63 NDF tasa impl 1M': dif('CHNI1M Curncy|PX_LAST'),
    '64 Vol impl CLP 1M (nivel)': b['USDCLPV1M Curncy|PX_LAST'],
    '65 Flujo derivados offshore': flujo_off.shift(2).diff().where(salto_ok),   # el BCCh publica con 2 días de rezago
    '66 Flujo derivados local': flujo_loc.shift(2).diff().where(salto_ok),
    '68 Y rezagada (apertura t-1 a t)': oo,
})

# Variable objetivo: Y = ln(apertura t+2 / apertura t+1); vacía si hay un hueco de más de 5 días entre t y t+2
Y = oo.shift(-2).where(salto_ok.shift(-1, fill_value=False) & salto_ok.shift(-2, fill_value=False))

In [5]:
# =============================================================================
# 4. Faltantes y filas finales
# =============================================================================
base = X.copy()
base['Flujos disponibles (indicador)'] = flujo_off.shift(2).notna().astype(int)
base['Y'] = Y
base.index.name = 'fecha'

niveles = [c for c in X.columns if 'nivel' in c]
cambios = [c for c in X.columns if c not in niveles]
base[cambios] = base[cambios].fillna(0)          # retornos y diferencias: mercado cerrado = sin cambio

descartar = base[niveles].isna().any(axis=1) | ~salto_ok                     # niveles faltantes y primer día tras un hueco
descartar |= base['Y'].isna() & (base.index <= base['Y'].last_valid_index())  # se conservan solo las filas finales sin Y (predicción)
base = base[~descartar]

# Días corridos que dura la posición (apertura de t+1 a apertura de t+2), para el carry
fechas = b.index.to_series()
auxiliares = pd.DataFrame({'dias_posicion': fechas.diff().dt.days.shift(-2)}).reindex(base.index)

# Precios del USD/CLP de todas las jornadas hábiles, para los indicadores técnicos del modelo
precios = pd.DataFrame({'apertura': O, 'maximo': H_aj, 'minimo': L_aj, 'cierre': Cl})
precios.index.name = 'fecha'

with pd.ExcelWriter(ARCHIVO_MODELO, engine='openpyxl') as w:
    base.reset_index().to_excel(w, sheet_name='datos', index=False)
    auxiliares.reset_index().to_excel(w, sheet_name='auxiliares', index=False)
    precios.reset_index().to_excel(w, sheet_name='precios', index=False)
print(f'Guardado {ARCHIVO_MODELO}: {base.shape[0]} filas, {base["Y"].notna().sum()} con Y, '
      f'última fecha {base.index.max().date()} (fila de predicción)')

Guardado base_modelo.xlsx: 5324 filas, 5322 con Y, última fecha 2026-11-06 (fila de predicción)
